# Portfolio Recommender Using ML Outputs

Goal: combine predicted alpha, existing quality scores, and current regime/risk signals into prototype conservative, balanced, and aggressive portfolios.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import numpy as np
import pandas as pd

from ai_models.feature_builder import build_feature_table
from ai_models.quality_score_model import run_quality_score_model
from ai_models.regime_detection_model import run_regime_detection_model
from ai_models.risk_detector import run_systemic_risk_detector

features = build_feature_table(
    fundamentals_path="data/fundamentals_cache.parquet",
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
).features.reset_index()
quality = run_quality_score_model(features)
regime = run_regime_detection_model(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")
risk = run_systemic_risk_detector(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")

In [ ]:
# Placeholder ML output until a learned model is added.
candidate = features.merge(quality, on="Ticker", how="left")
candidate["PredictedAlpha"] = (
    candidate["QualityScore"].fillna(0.0) / 100.0
    + candidate["Momentum_12M"].fillna(0.0) * 0.5
    - candidate["Volatility_63D"].fillna(0.0) * 0.2
)
candidate[["Ticker", "QualityScore", "Momentum_12M", "Volatility_63D", "PredictedAlpha"]].head()

In [ ]:
latest_regime = regime.iloc[-1].to_dict() if not regime.empty else {}
latest_risk = risk.iloc[-1].to_dict() if not risk.empty else {}
latest_regime, latest_risk

In [ ]:
conservative = candidate.sort_values(["QualityScore", "Volatility_63D"], ascending=[False, True]).head(10)
balanced = candidate.sort_values(["PredictedAlpha", "QualityScore"], ascending=[False, False]).head(10)
aggressive = candidate.sort_values(["PredictedAlpha", "Momentum_12M"], ascending=[False, False]).head(10)

conservative[["Ticker", "PredictedAlpha", "QualityScore", "Volatility_63D"]]